<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/HunyuanVideo-Foley_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/HunyuanVideo-Foley_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎬 HunyuanVideo-Foley — 48 kHz Representation-Aligned Video Foley & SFX Studio (`XL` & `XXL`)

<p align="left">
  <a href="https://github.com/Tencent-Hunyuan/HunyuanVideo-Foley"><img src="https://img.shields.io/badge/GitHub-Tencent--Hunyuan%2FHunyuanVideo--Foley-181717?logo=github" alt="GitHub"></a>
  <a href="https://huggingface.co/tencent/HunyuanVideo-Foley"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-tencent%2FHunyuanVideo--Foley-amber" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Audio-48%20kHz%20Hi--Fi%20DAC%20VAE-success" alt="48kHz">
  <img src="https://img.shields.io/badge/VRAM-8%20GB%20(XL%20%2B%20Offload)%20%E2%86%92%20T4%20Ready-orange" alt="T4 Ready">
</p>

Welcome to the **Tencent HunyuanVideo-Foley** Google Colab studio! Built on [`Tencent-Hunyuan/HunyuanVideo-Foley`](https://github.com/Tencent-Hunyuan/HunyuanVideo-Foley) (`arXiv:2508.16930`), `HunyuanVideo-Foley` achieves SOTA audio-visual synchronization and 48 kHz acoustic fidelity:

- **🎛️ Dual Checkpoint Support (`XL` & `XXL`) with CPU Offload** — Run the memory-efficient **`XL` model (`hunyuanvideo_foley_xl.pth`, ~8 GB VRAM with offload)** on free Colab T4 GPUs, or switch to the flagship **`XXL` model (`hunyuanvideo_foley.pth`, ~12 GB VRAM with offload)** on L4/A100.
- **🎬 Frame-Accurate 48 kHz Video-to-Foley (`Synchformer`)** — Upload any silent video or select directly from your Google Drive exports (`Wan2.2`, `Wan-VACE`, `FramePack`, `SeedVR2`, `LTX-2`) and generate up to **4 multi-take 48 kHz Foley candidates** per run!
- **🎚️ Original Video Audio + AI Foley Blender** — Mix the newly synthesized 48 kHz Foley stem with the source video's existing dialogue/music track at custom dB levels.
- **🪟 Text-Guided 48 kHz SFX & Windows Sound Event Pack Builder** — Automatically synthesizes a motion-paced conditioning canvas behind the scenes so you can harness `HunyuanVideo-Foley`'s 48 kHz DAC VAE for standalone text-to-SFX and **48 kHz Windows 10/11 Sound Schemes**!

In [ ]:
#@title Step 1: Verify GPU & Install HunyuanVideo-Foley Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

REPO_DIR = "/content/HunyuanVideo-Foley"

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — GPU runtime is required.")

    if not os.path.exists(REPO_DIR):
        run_cmd(f"git clone --depth 1 https://github.com/Tencent-Hunyuan/HunyuanVideo-Foley.git {REPO_DIR}", check=True)

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    req_file = os.path.join(REPO_DIR, "requirements.txt")
    if os.path.exists(req_file):
        run_cmd(f"{sys.executable} -m pip install -q -r {req_file}", check=False)

    pip_pkgs = (
        "einops safetensors soundfile huggingface_hub>=0.26.0 "
        "transformers accelerate omegaconf loguru decord av moviepy "
        "descript-audio-codec open-clip-torch timm gradio>=5.0.0 matplotlib scipy librosa tqdm"
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    clear_output()
    print("✅ Step 1 Complete: HunyuanVideo-Foley dependencies installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Model Cache
import os
import sys

WORKSPACE_DIR = "/content/hunyuan_foley_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted: {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/hunyuanvideo_foley"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/HunyuanVideo_Foley"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")

MODEL_DIR = os.path.join(CACHE_ROOT, "pretrained_models")
for d in (CACHE_ROOT, OUTPUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.environ["HIFI_FOLEY_MODEL_PATH"] = MODEL_DIR
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print(f"📂 Model Cache   : {MODEL_DIR}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core HunyuanVideo-Foley Engine (XL & XXL) & Blank-Canvas SFX Helper
import os
import gc
import sys
import time
import glob
import subprocess
import torch
import torchaudio
import numpy as np
import soundfile as sf
from huggingface_hub import hf_hub_download

REPO_DIR = "/content/HunyuanVideo-Foley"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


def discover_drive_videos():
    roots = ["/content/drive/MyDrive", "/content"]
    found = []
    for r in roots:
        if os.path.exists(r):
            for ext in ("*.mp4", "*.mov", "*.webm"):
                found.extend(glob.glob(os.path.join(r, "**", ext), recursive=True))
    found = sorted(set(found), key=lambda p: os.path.getmtime(p) if os.path.exists(p) else 0, reverse=True)
    return ["(Upload video below)"] + found[:40]


def create_blank_pacing_video(duration_s: float = 3.0, fps: int = 16) -> str:
    """Create a synthetic visual transient pulse MP4 at t=0.04s so Synchformer triggers a crisp onset for standalone Text-to-SFX."""
    dur = max(1.0, min(float(duration_s), 15.0))
    out_mp4 = os.path.join("/tmp", f"hv_canvas_{uuid.uuid4().hex[:8]}.mp4")
    cmd = (
        f'ffmpeg -y -f lavfi -i "color=c=0x38bdf8:s=256x256:r={int(fps)}:d={dur:.2f},'
        f'fade=t=in:st=0.02:d=0.04,fade=t=out:st=0.08:d={max(0.3, dur - 0.12):.2f}" '
        f'-c:v libx264 -pix_fmt yuv420p -an "{out_mp4}"'
    )
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not os.path.exists(out_mp4):
        cmd_fb = (
            f'ffmpeg -y -f lavfi -i "color=c=0x1e293b:s=256x256:r={int(fps)}:d={dur:.2f}" '
            f'-c:v libx264 -pix_fmt yuv420p -an "{out_mp4}"'
        )
        subprocess.run(cmd_fb, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_mp4


class HunyuanFoleyStudio:
    """Manager for Tencent HunyuanVideo-Foley ('xl' and 'xxl' with CPU offload)."""

    def __init__(self):
        self.model_dict = None
        self.cfg = None
        self.current_key = None
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def unload(self):
        self.model_dict = None
        self.cfg = None
        self.current_key = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def ensure_weights(self, model_size: str):
        shared_files = ["synchformer_state_dict.pth", "vae_128d_48k.pth"]
        size_files = (
            ["hunyuanvideo_foley_xl.pth", "config_xl.yaml"]
            if model_size == "xl"
            else ["hunyuanvideo_foley.pth", "config.yaml"]
        )
        for fname in shared_files + size_files:
            target = os.path.join(MODEL_DIR, fname)
            if not os.path.exists(target):
                print(f"⬇️ Downloading {fname} from tencent/HunyuanVideo-Foley...")
                hf_hub_download(
                    repo_id="tencent/HunyuanVideo-Foley",
                    filename=fname,
                    local_dir=MODEL_DIR,
                )

    def ensure_loaded(self, model_size: str = "xl", enable_offload: bool = True) -> str:
        size_clean = "xl" if "xl" in model_size.lower() and "xxl" not in model_size.lower() else "xxl"
        key = f"{size_clean}_offload{enable_offload}"
        if self.current_key == key and self.model_dict is not None:
            return f"✅ Using cached `HunyuanVideo-Foley ({size_clean.upper()})` (Offload={enable_offload})."

        self.unload()
        self.ensure_weights(size_clean)

        from hunyuanvideo_foley.utils.model_utils import load_model
        cfg_rel = "configs/hunyuanvideo-foley-xl.yaml" if size_clean == "xl" else "configs/hunyuanvideo-foley-xxl.yaml"
        cfg_path = os.path.join(REPO_DIR, cfg_rel)
        if not os.path.exists(cfg_path):
            cfg_path = os.path.join(MODEL_DIR, "config_xl.yaml" if size_clean == "xl" else "config.yaml")

        self.model_dict, self.cfg = load_model(
            MODEL_DIR,
            cfg_path,
            self.device,
            enable_offload=bool(enable_offload),
            model_size=size_clean,
        )
        self.current_key = key
        return f"✅ Loaded `HunyuanVideo-Foley ({size_clean.upper()})` on `{self.device}` (Offload={enable_offload})."

    @torch.inference_mode()
    def infer_video(
        self,
        video_path: str,
        prompt: str = "",
        neg_prompt: str = "low quality, harsh noise, distortion, static",
        model_size: str = "xl",
        enable_offload: bool = True,
        guidance_scale: float = 4.5,
        steps: int = 50,
        sample_nums: int = 1,
        seed: int = 42,
    ):
        msg = self.ensure_loaded(model_size=model_size, enable_offload=enable_offload)
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        torch.manual_seed(actual_seed)

        from hunyuanvideo_foley.utils.feature_utils import feature_process
        from hunyuanvideo_foley.utils.model_utils import denoise_process

        visual_feats, text_feats, audio_len_in_s = feature_process(
            video_path,
            (prompt or "").strip(),
            self.model_dict,
            self.cfg,
            neg_prompt=neg_prompt,
        )
        audio_batch, sample_rate = denoise_process(
            visual_feats,
            text_feats,
            audio_len_in_s,
            self.model_dict,
            self.cfg,
            guidance_scale=float(guidance_scale),
            num_inference_steps=int(steps),
            batch_size=int(sample_nums),
        )
        # audio_batch shape: [B, C, T]
        takes = []
        for i in range(audio_batch.shape[0]):
            arr = audio_batch[i].detach().cpu().float().numpy().squeeze()
            if arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
                arr = arr.T
            takes.append(arr)
        return takes, int(sample_rate), actual_seed, msg


HV_STUDIO = HunyuanFoleyStudio()
print("✅ Step 3 Complete: HunyuanFoleyStudio initialized!")

In [ ]:
#@title Step 4: Launch HunyuanVideo-Foley Gradio Studio (4 Tabs)
import os
import time
import subprocess
import gradio as gr
from IPython.display import clear_output

HV_MODELS = [
    "xl (HunyuanVideo-Foley-XL — Memory Friendly, ~8 GB VRAM with Offload, T4 Ready)",
    "xxl (HunyuanVideo-Foley-XXL — Flagship Quality, ~12 GB VRAM with Offload)",
]


def ui_hv_video_foley(
    model_choice,
    enable_offload,
    drive_video,
    upload_video,
    prompt,
    neg_prompt,
    guidance_scale,
    steps,
    sample_nums,
    seed,
    blend_orig_audio,
    orig_mix_db,
):
    try:
        vid_path = upload_video if (upload_video and os.path.exists(upload_video)) else (
            drive_video if (drive_video and os.path.exists(drive_video)) else None
        )
        if not vid_path:
            return None, None, None, None, "❌ Please upload a video or pick one from Google Drive."
        t0 = time.time()
        takes, sr, used_seed, load_msg = HV_STUDIO.infer_video(
            video_path=vid_path,
            prompt=prompt,
            neg_prompt=neg_prompt,
            model_size=model_choice,
            enable_offload=bool(enable_offload),
            guidance_scale=float(guidance_scale),
            steps=int(steps),
            sample_nums=int(sample_nums),
            seed=int(seed),
        )
        ts = int(time.time())
        wav_paths = []
        for i, arr in enumerate(takes):
            peak = float(np.max(np.abs(arr)))
            if peak > 1e-6:
                arr = arr / peak * 0.90
            w_path = os.path.join(OUTPUT_DIR, f"hv_foley_{ts}_s{used_seed}_take{i+1}.wav")
            save_pcm16_wav(w_path, arr, sr)
            wav_paths.append(w_path)

        out_mp4 = os.path.join(OUTPUT_DIR, f"hv_foley_muxed_{ts}_s{used_seed}.mp4")
        muxed_ok = False
        if blend_orig_audio:
            mix_cmd = (
                f'ffmpeg -y -i "{vid_path}" -i "{wav_paths[0]}" '
                f'-filter_complex "[0:a]volume={float(orig_mix_db):.1f}dB[a0];[1:a]volume=0dB[a1];'
                f'[a0][a1]amix=inputs=2:duration=first:dropout_transition=0[aout]" '
                f'-map 0:v:0 -map "[aout]" -c:v copy -c:a aac -b:a 256k -shortest "{out_mp4}"'
            )
            res_mix = subprocess.run(mix_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            muxed_ok = (res_mix.returncode == 0 and os.path.exists(out_mp4))

        if not muxed_ok:
            from hunyuanvideo_foley.utils.media_utils import merge_audio_video
            try:
                merge_audio_video(wav_paths[0], vid_path, out_mp4)
            except Exception:
                cmd = f'ffmpeg -y -i "{vid_path}" -i "{wav_paths[0]}" -c:v copy -map 0:v:0 -map 1:a:0 -shortest "{out_mp4}"'
                subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        spec_img = render_waveform_and_spectrogram(wav_paths[0], title=f"HunyuanVideo-Foley ({sr} Hz, Take 1)")
        take2 = wav_paths[1] if len(wav_paths) > 1 else None
        status = (
            f"✅ Generated **{len(wav_paths)}** 48 kHz Foley take(s) in **{time.time()-t0:.1f}s** (Seed `{used_seed}`).\n"
            f"- {load_msg}\n- Muxed video saved to: `{out_mp4}`"
        )
        return out_mp4 if os.path.exists(out_mp4) else vid_path, wav_paths[0], take2, spec_img, status
    except Exception as e:
        return None, None, None, None, f"❌ HunyuanVideo-Foley Error: {e}"


def ui_hv_text_sfx(
    model_choice,
    enable_offload,
    prompt,
    neg_prompt,
    duration,
    guidance_scale,
    steps,
    seed,
    apply_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        if not prompt or not prompt.strip():
            return None, None, "❌ Please enter a sound effect prompt."
        t0 = time.time()
        canvas_mp4 = create_blank_pacing_video(duration_s=float(duration), fps=16)
        takes, sr, used_seed, load_msg = HV_STUDIO.infer_video(
            video_path=canvas_mp4,
            prompt=prompt.strip(),
            neg_prompt=neg_prompt,
            model_size=model_choice,
            enable_offload=bool(enable_offload),
            guidance_scale=float(guidance_scale),
            steps=int(steps),
            sample_nums=1,
            seed=int(seed),
        )
        arr = takes[0]
        if apply_mastering:
            arr = master_ui_sound(
                arr, sr, target_duration_s=float(duration), trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
        out_wav = os.path.join(OUTPUT_DIR, f"hv_sfx_48k_{int(time.time())}_s{used_seed}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec_img = render_waveform_and_spectrogram(out_wav, title=f"Hunyuan 48 kHz Standalone SFX ({duration}s)")
        return out_wav, spec_img, f"✅ Generated 48 kHz SFX in **{time.time()-t0:.1f}s** (`{out_wav}`)"
    except Exception as e:
        return None, None, f"❌ Standalone SFX Error: {e}"


def ui_hv_windows_pack(
    model_choice,
    enable_offload,
    scheme_name,
    theme_name,
    custom_style,
    selected_events,
    guidance_scale,
    steps,
    base_seed,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Select at least one Windows system event."
        t0 = time.time()
        pack_dir = os.path.join(OUTPUT_DIR, f"hv_winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        first_wav = None
        seed_val = int(base_seed) if int(base_seed) >= 0 else 42

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            dur = float(ev_meta["duration"])
            ev_prompt = build_windows_event_prompt(ev_key, theme_name, custom_style)
            canvas_mp4 = create_blank_pacing_video(duration_s=max(dur + 0.5, 1.5), fps=16)
            takes, sr, _, _ = HV_STUDIO.infer_video(
                video_path=canvas_mp4,
                prompt=ev_prompt,
                model_size=model_choice,
                enable_offload=bool(enable_offload),
                guidance_scale=float(guidance_scale),
                steps=int(steps),
                sample_nums=1,
                seed=seed_val + idx,
            )
            mastered = master_ui_sound(
                takes[0], sr, target_duration_s=dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
            if first_wav is None:
                first_wav = fpath

        model_lbl = f"HunyuanVideo-Foley ({model_choice.split(' ')[0].upper()} 48 kHz)"
        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label=model_lbl,
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label=model_lbl,
            embed_base64=True,
        )
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"48 kHz Event: {selected_events[0]}") if first_wav else None
        return zip_path, first_wav, spec_img, board_html, (
            f"✅ Packaged **{len(event_wav_map)}** 48 kHz events into `{os.path.basename(zip_path)}` in **{time.time()-t0:.1f}s**!"
        )
    except Exception as e:
        return None, None, None, "", f"❌ Windows Sound Pack Error: {e}"


def ui_hv_reroll_single_event(
    model_choice,
    enable_offload,
    scheme_name,
    theme_name,
    custom_style,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    guidance_scale,
    steps,
    trim_db,
    fade_out_ms,
):
    try:
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        dur = float(ev_meta["duration"])
        ev_prompt = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, custom_style)
        canvas_mp4 = create_blank_pacing_video(duration_s=max(dur + 0.5, 1.5), fps=16)
        takes, sr, _, _ = HV_STUDIO.infer_video(
            video_path=canvas_mp4,
            prompt=ev_prompt,
            model_size=model_choice,
            enable_offload=bool(enable_offload),
            guidance_scale=float(guidance_scale),
            steps=int(steps),
            sample_nums=1,
            seed=int(reroll_seed),
        )
        mastered = master_ui_sound(
            takes[0], sr, target_duration_s=dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=ev_prompt,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=theme_name,
            model_label=f"HunyuanVideo-Foley ({model_choice.split(' ')[0].upper()} 48 kHz)",
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


with gr.Blocks(title="Tencent HunyuanVideo-Foley — 48 kHz Video Foley & SFX Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🎬 Tencent HunyuanVideo-Foley — 48 kHz Synchronized Video Foley & SFX Studio\n"
        "Powered by `tencent/HunyuanVideo-Foley` (`XL` & `XXL` models, 48 kHz DAC VAE + `Synchformer`)."
    )
    banner = gr.Markdown("⏳ Initializing HunyuanVideo-Foley studio...")

    with gr.Row():
        hv_model = gr.Dropdown(
            choices=HV_MODELS,
            value=HV_MODELS[0],
            label="🎛️ HunyuanVideo-Foley Checkpoint Size",
            info="'xl' uses ~8 GB VRAM with offload (fits free Colab T4); 'xxl' uses ~12 GB with offload.",
            scale=3,
        )
        hv_offload = gr.Checkbox(
            value=True,
            label="🧠 Enable CPU Offload (Saves ~8 GB GPU VRAM)",
            info="Keep enabled on 16 GB T4 GPUs; disable on A100 40GB for faster inference.",
            scale=1,
        )

    with gr.Tabs():
        with gr.Tab("🎬 1. 48 kHz Synchronized Video-to-Foley Studio"):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        hv_drive_vid = gr.Dropdown(
                            choices=discover_drive_videos(),
                            value="(Upload video below)",
                            label="📂 Cross-Notebook Google Drive Video Picker",
                            info="Select any video from your Drive exports (Wan2.2, Wan-VACE, FramePack, SeedVR2).",
                            scale=3,
                        )
                        hv_ref_btn = gr.Button("🔄 Refresh Drive", scale=1)
                    hv_upload_vid = gr.Video(label="🎬 Or Upload Silent / Source Video (.mp4 / .mov)")
                    hv_prompt = gr.Textbox(
                        value="Crisp realistic foley sound effects synchronized with the visual motion",
                        lines=2,
                        label="📝 Foley Text Prompt (Optional — Guides Acoustic Style & Materials)",
                        info="Describe specific materials or actions (or leave general to let visual features lead).",
                    )
                    hv_neg = gr.Textbox(
                        value="low quality, harsh noise, distortion, muffled, clipping",
                        label="🚫 Negative Prompt",
                        info="Qualities to suppress.",
                    )
                    with gr.Row():
                        hv_cfg = gr.Slider(1.0, 9.0, value=4.5, step=0.5, label="🧭 Guidance Scale", info="Default: 4.5.")
                        hv_steps = gr.Slider(15, 100, value=50, step=5, label="⚡ Denoising Steps", info="Default: 50 (or 30 for faster previews).")
                        hv_takes = gr.Slider(1, 3, value=2, step=1, label="🎲 Multi-Take Candidates (`sample_nums`)", info="Generate 1–3 synchronized takes in a single batch.")
                        hv_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Random seed.")
                    with gr.Row():
                        hv_blend_orig = gr.Checkbox(value=False, label="🎚️ Blend Original Video Audio Track with AI Foley", info="Mixes the source video's existing audio track with the newly synthesized 48 kHz Foley stem.")
                        hv_orig_db = gr.Slider(-30.0, 0.0, value=-6.0, step=1.0, label="🎚️ Original Audio Track Mix Level (dB)", info="Attenuation applied to the original video audio when blending.")
                    hv_btn = gr.Button("🎬 Generate 48 kHz Synchronized Foley", variant="primary")

                with gr.Column(scale=2):
                    hv_out_mp4 = gr.Video(label="🎥 Video Muxed with 48 kHz Foley (Take 1)")
                    hv_take1 = gr.Audio(label="🎧 48 kHz Foley Stem — Take 1", type="filepath")
                    hv_take2 = gr.Audio(label="🎧 48 kHz Foley Stem — Take 2 (Variation)", type="filepath")
                    hv_spec = gr.Image(label="📊 48 kHz Waveform & Spectrogram", type="filepath")
                    hv_status = gr.Markdown()

            hv_ref_btn.click(fn=lambda: gr.update(choices=discover_drive_videos()), inputs=None, outputs=[hv_drive_vid])
            hv_btn.click(
                fn=ui_hv_video_foley,
                inputs=[hv_model, hv_offload, hv_drive_vid, hv_upload_vid, hv_prompt, hv_neg, hv_cfg, hv_steps, hv_takes, hv_seed, hv_blend_orig, hv_orig_db],
                outputs=[hv_out_mp4, hv_take1, hv_take2, hv_spec, hv_status],
            )

        with gr.Tab("🔊 2. Standalone 48 kHz Text-to-SFX Designer"):
            gr.Markdown(
                "### 🔊 Standalone Text-to-SFX via 48 kHz DAC VAE\n"
                "Synthesize pure sound effects without needing a video — the studio generates a temporary pacing canvas automatically."
            )
            with gr.Row():
                with gr.Column():
                    t_prompt = gr.Textbox(
                        value="Crisp crystalline glass marimba two-note ascending notification chime, studio foley",
                        lines=3,
                        label="📝 Sound Effect Prompt",
                        info="Describe the sound effect to synthesize at 48 kHz.",
                    )
                    t_neg = gr.Textbox(value="low quality, noise, distortion", label="🚫 Negative Prompt", info="Artifacts to avoid.")
                    with gr.Row():
                        t_dur = gr.Slider(1.0, 15.0, value=3.0, step=0.5, label="⏱️ Duration (Seconds)", info="1s–15s.")
                        t_cfg = gr.Slider(1.0, 9.0, value=4.5, step=0.5, label="🧭 CFG Scale", info="Default 4.5.")
                        t_steps = gr.Slider(15, 80, value=40, step=5, label="⚡ Steps", info="Diffusion steps.")
                        t_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Seed.")
                    with gr.Row():
                        t_master = gr.Checkbox(value=True, label="✨ Zero-Latency UI Mastering", info="Auto-trim leading silence.")
                        t_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim (dB)", info="Silence threshold.")
                        t_fade = gr.Slider(20.0, 350.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth release.")
                    t_btn = gr.Button("🚀 Synthesize 48 kHz Sound Effect", variant="primary")
                with gr.Column():
                    t_wav = gr.Audio(label="🎧 48 kHz SFX Output", type="filepath")
                    t_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    t_status = gr.Markdown()

            t_btn.click(
                fn=ui_hv_text_sfx,
                inputs=[hv_model, hv_offload, t_prompt, t_neg, t_dur, t_cfg, t_steps, t_seed, t_master, t_trim, t_fade],
                outputs=[t_wav, t_spec, t_status],
            )

        with gr.Tab("🪟 3. 48 kHz Windows Sound Event Pack Builder"):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        hw_scheme = gr.Textbox(value="Hunyuan_Foley_48k", label="🏷️ Windows Sound Scheme Name", info="Scheme name in Windows Sound Control Panel.")
                        hw_theme = gr.Dropdown(
                            choices=list(SONIC_THEME_PRESETS.keys()),
                            value=list(SONIC_THEME_PRESETS.keys())[0],
                            label="🎨 Sonic Theme Preset",
                            info="Cohesive theme across all system events.",
                        )
                    hw_custom = gr.Textbox(value="", label="✍️ Custom Style Override (Optional)", info="Leave blank to use preset.")
                    hw_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS[:6],
                        label="🔔 Windows System Events to Generate (48 kHz)",
                        info="Select events to include in the pack.",
                    )
                    with gr.Row():
                        hw_cfg = gr.Slider(1.0, 9.0, value=4.5, step=0.5, label="🧭 CFG Scale", info="Default 4.5.")
                        hw_steps = gr.Slider(15, 60, value=35, step=5, label="⚡ Steps per Event", info="35 steps for fast pack generation.")
                        hw_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Seed.")
                    with gr.Row():
                        hw_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim (dB)", info="Zero-latency trim.")
                        hw_fade = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Release tail.")
                    hw_btn = gr.Button("🪟 Build 48 kHz Windows Sound Scheme (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            hw_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Pick a single 48 kHz event to regenerate.")
                            hw_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Fresh seed.")
                            hw_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Semitone shift.")
                        hw_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt override for this event...", label="✍️ Single-Event Prompt Override (Optional)", info="Leave blank to use theme prompt.")
                        hw_rr_btn = gr.Button("🔄 Re-Roll Single Event & Update 48 kHz Pack")
                with gr.Column(scale=2):
                    hw_zip = gr.File(label="📦 Download 48 kHz Windows Sound Pack (.zip)")
                    hw_prev = gr.Audio(label="🎧 Sample Event Preview", type="filepath")
                    hw_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    hw_status = gr.Markdown()
            hw_board = gr.HTML(label="🎹 Interactive 48 kHz Windows Sound Scheme Audition Board")

            hw_btn.click(
                fn=ui_hv_windows_pack,
                inputs=[hv_model, hv_offload, hw_scheme, hw_theme, hw_custom, hw_events, hw_cfg, hw_steps, hw_seed, hw_trim, hw_fade],
                outputs=[hw_zip, hw_prev, hw_spec, hw_board, hw_status],
            )
            hw_rr_btn.click(
                fn=ui_hv_reroll_single_event,
                inputs=[hv_model, hv_offload, hw_scheme, hw_theme, hw_custom, hw_rr_ev, hw_rr_prompt, hw_rr_seed, hw_rr_pitch, hw_cfg, hw_steps, hw_trim, hw_fade],
                outputs=[hw_zip, hw_prev, hw_spec, hw_board, hw_status],
            )

    demo.load(
        fn=lambda: f"🟢 **Tencent HunyuanVideo-Foley (48 kHz XL & XXL) Studio Ready** on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`!",
        inputs=None,
        outputs=[banner],
    )

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — Standalone 48 kHz Foley SFX Synthesis
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_PROMPT = "Crisp crystalline glass marimba two-note ascending notification chime, studio foley" #@param {type:"string"}
QUICKTEST_DURATION = 2.5 #@param {type:"number"}
QUICKTEST_SEED = 42 #@param {type:"integer"}

try:
    wav_path, _, msg = ui_hv_text_sfx(
        model_choice="xl",
        enable_offload=True,
        prompt=QUICKTEST_PROMPT,
        neg_prompt="noise, distortion",
        duration=QUICKTEST_DURATION,
        guidance_scale=4.5,
        steps=35,
        seed=QUICKTEST_SEED,
        apply_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch Video Foley & Windows Sound Pack Export
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "Hunyuan_48k_Scheme" #@param {type:"string"}
BATCH_THEME = "Cozy Wooden Kalimba & Mallet" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 42 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_hv_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS[:6], desc="Hunyuan 48 kHz Windows Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            dur = float(ev_meta["duration"])
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            canvas_mp4 = create_blank_pacing_video(duration_s=max(dur + 0.5, 1.5), fps=16)
            takes, sr, _, _ = HV_STUDIO.infer_video(
                video_path=canvas_mp4,
                prompt=prompt,
                model_size="xl",
                enable_offload=True,
                guidance_scale=4.5,
                steps=30,
                sample_nums=1,
                seed=BATCH_SEED + idx,
            )
            mastered = master_ui_sound(takes[0], sr, target_duration_s=dur)
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": prompt}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "HunyuanVideo-Foley XL (48 kHz)", OUTPUT_DIR)
        print(f"✅ Packaged Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")